# Phase 3: Deep Q-Network (DQN) Dynamic Pricing

This notebook trains a PyTorch-based Deep Q-Network (DQN) agent on the `HotelPricingEnvironment` and compares its performance against Tabular Q-Learning and baselines (Fixed, Discount, and Random Pricing).

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

# Ensure project root is in path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import FEATURES_DATA_PATH, REPORTS_DIR, FIGURES_DIR
print(f"Project root path verified: {project_root}")
print(f"Features dataset path: {FEATURES_DATA_PATH}")

### 1. Load Processed Dataset
Verify features are engineered and loaded correctly.

In [ ]:
df_features = pd.read_csv(FEATURES_DATA_PATH)
print(f"Loaded features dataset shape: {df_features.shape}")
df_features.head()

### 2. Train DQN Agent
We run the DQN training flow which fits the simulator, sets up the environment, initializes the PyTorch model and memory buffer, executes optimization steps, and saves metrics and checkpoint files.

In [ ]:
from src.dqn.trainer import run_dqn_training_flow
import time

print("Starting DQN Training Flow...")
start_time = time.time()
agent, env, simulator, dqn_time = run_dqn_training_flow(episodes=600)
print(f"DQN Training successfully completed in {dqn_time:.2f} seconds!")

### 3. Display Training Curves
Show the reward convergence and neural network training loss decay.

In [ ]:
print("DQN Learning Curve (Episode rewards and 50-episode rolling average):")
display(Image(filename=str(FIGURES_DIR / "dqn_learning_curve.png")))

print("DQN MSE Training Loss Curve (log scale):")
display(Image(filename=str(FIGURES_DIR / "loss_curve.png")))

### 4. Comparative Evaluation
Evaluate the trained DQN agent against pre-trained Q-Learning and baselines over 100 benchmark episodes.

In [ ]:
from src.dqn.evaluator import run_dqn_evaluation_flow
from src.rl.trainer import run_training_flow as run_q_training_flow

# Ensure Tabular Q-learning is trained to compare training times
print("Ensuring Tabular Q-Learning training metrics are loaded...")
q_start = time.time()
q_agent, _, _ = run_q_training_flow()
q_time = time.time() - q_start

print("Running comparative evaluation flow...")
df_comparison = run_dqn_evaluation_flow(
    dqn_agent=agent,
    env=env,
    dqn_training_time=dqn_time,
    q_learning_training_time=q_time
)

# Display summary table
df_comparison

### 5. Display Evaluation Benchmarks
View performance comparisons across agents.

In [ ]:
print("Total Revenue Comparison:")
display(Image(filename=str(FIGURES_DIR / "dqn_revenue_comparison.png")))

print("Occupancy Rate Comparison:")
display(Image(filename=str(FIGURES_DIR / "dqn_occupancy_comparison.png")))

print("Price Distributions Boxplot:")
display(Image(filename=str(FIGURES_DIR / "dqn_price_distribution.png")))

print("DQN Predicted Q-Value Diagnostic Histogram:")
display(Image(filename=str(FIGURES_DIR / "q_value_histogram.png")))